Foundry Skills live in git. That's great for developers, but most of the
people who actually know whether a skill is *right* ? a support lead who
knows the real refund policy, a product manager who knows what a loyalty
program should say ? don't use git, don't have a GitHub habit, and
shouldn't have to learn one just to approve a two-sentence wording change.

This recipe walks through a sample we built to close that gap: a Foundry
hosted agent, published to Microsoft Teams, that lets business users
review skill pull requests, comment, approve, merge, and propose their
own changes ? all from a normal chat, with every action attributed to
*their own* GitHub account.


## 1 / Why we built this

Skills are code review artifacts whether we like it or not. If you store
Foundry Skills as files in a repo (which is the natural place for them ?
versioned, diffable, revertable), then changing a skill means opening a
pull request, and approving a skill means approving that PR. That's fine
for engineers. It's a wall for everyone else.

The obvious first fix ? "let an agent do it for them" ? has an obvious
trap. If the agent uses one shared GitHub token to list PRs, comment,
approve, and merge on behalf of *every* Teams user, you've built a system
where the audit trail says "the bot did it," not "Alice approved it" or
"Bob proposed it." That's not just unsatisfying, it's actually unsafe:
GitHub's own protections ? "require an approving review," "you can't
approve your own PR" ? are keyed off the identity making the API call. A
shared token can't be checked by two different people, and it can approve
its own proposals all day long. We tried building it that way first. It
worked, technically, and then we realized it worked around the one
guarantee that actually matters.

So the real requirement wasn't "let business users talk to GitHub through
an agent." It was: **every git action a Teams user takes through this bot
must be authenticated as that person, on GitHub, with no exceptions** ?
and the bot itself should never need to know or store anyone's GitHub
credentials.


## 2 / How we built it

### The shape of it

```text
Business user in Teams
  -> Foundry hosted agent (Agent Framework, Python)
  -> Foundry toolbox (MCP) -> GitHub's catalog MCP server
       https://api.githubcopilot.com/mcp, OAuth2, per Teams user
```

The agent itself is a small Agent Framework `Agent`. It has exactly one
tool: a `FoundryToolbox` pointed at a Foundry **project connection** that
targets GitHub's own MCP server. No custom REST wrappers, no
`GITHUB_TOKEN`, no bot identity at all.


In [ ]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient, FoundryToolbox, ResponsesHostServer

# Resolved from FOUNDRY_PROJECT_ENDPOINT + TOOLBOX_NAME env vars.
toolbox = FoundryToolbox(credential)

agent = Agent(
    client=client,
    instructions=INSTRUCTIONS,  # see section 2 below - plain-language, no git jargon
    tools=toolbox,
)


### Per-user auth, for free

Here's the part that made this worth writing up. `FoundryToolbox` forwards
a platform call-id header on every request. The Foundry MCP proxy uses
that header to resolve *which Teams user* is making the call, and swaps in
that user's own GitHub OAuth token before hitting
`api.githubcopilot.com/mcp`. We didn't write any of that plumbing. The
agent code has zero awareness of who's asking ? it just calls
`gh___pull_request_read`, `gh___merge_pull_request`,
`gh___create_pull_request`, and so on, and the identity attached to each
call is whoever is actually chatting.

The connection itself is a standard catalog-MCP, managed-OAuth project
connection:


In [ ]:
connection_body = {
    "properties": {
        "authType": "OAuth2",
        "category": "RemoteTool",
        "target": "https://api.githubcopilot.com/mcp",
        "connectorName": "foundrygithubmcp",
        "metadata": {
            "type": "catalog_MCP",
            "toolEntityId": (
                "azureml://location/eastus/apiCenter/connectors-registry-prod-bl"
                "/type/tools/objectId/github/version/1"
            ),
        },
    }
}
# PUT this to
# {ARM}/.../projects/{proj}/connections/{name}?api-version=2025-04-01-preview


wired into a toolbox that the agent references by name. Consent isn't
something you call manually ? the first time a given Teams user's message
needs GitHub, the toolbox call comes back with a `CONSENT_REQUIRED` error
carrying a one-time GitHub sign-in link. The agent just shows that link to
the user. Once they sign in, every following call in that conversation ?
and every future conversation ? runs as them. We verified this the
boring, reliable way: by calling `gh___get_me` through the toolbox and
getting back the signer's own GitHub login, not a shared service account.

### The merge gate had to move

The nice thing about writing your own REST wrappers is you can put a gate
in front of anything ? e.g. "check for an approving review before I'll
even call the merge endpoint." The moment GitHub access becomes a generic
MCP toolbox instead of your own Python functions, that code-level
interception disappears; the model is calling GitHub's tools directly.

We replaced it with two layers instead of one:

1. **Instructions**, telling the model to read a PR's reviews
   (`gh___pull_request_read`) before ever calling
   `gh___merge_pull_request`, and to refuse if there's no approval or an
   outstanding "request changes."
2. **GitHub branch protection**, which is the layer that actually can't be
   talked around. "Require a pull request review before merging" makes
   GitHub itself reject the call regardless of what the model decides.
   GitHub also refuses self-approval outright ("you can't approve your
   own pull request") ? another guarantee that only works because the
   call is attributed to a real person.

Software-only enforcement (1) is a nice UX layer. Enforcement (2) is the
one you should actually rely on. That's worth saying plainly, because it's
tempting to assume "the agent's instructions say so" is a security
control. It isn't ? treat it as a courtesy on top of GitHub's real rules.

### Talking to business users like they're business users

The other lesson, once we started testing this with a non-developer
mindset: don't make them speak git. Early drafts of the instructions still
asked things like "what should I name the branch?" or "what's the PR
title?" ? exactly the kind of question that stops a support lead cold.
The fix was to push all of that down into the agent's own judgment:

- Ask only about *intent* ? which skill, what should change ? never about
  branch names, commit messages, or PR titles. The agent generates all of
  those itself from the skill name and a short summary.
- If someone asks what "merge" or "pull request" or "branch" means,
  explain it in one plain sentence, and only when asked or clearly needed
  ? not as an unprompted lecture.
- Don't guess at things the underlying GitHub tools don't actually support
  (we initially told the model to pass a `labels` argument straight into
  `create_pull_request`; that tool has no such parameter, only
  `reviewers` ? the actual GitHub MCP tool schema is the source of truth,
  not what seems like it should exist). Apply labels afterwards with
  `gh___issue_write` instead, since a pull request is addressable as an
  issue for labeling purposes.

None of this needed new code. It's all in the system instructions. Which
is the point: once the toolbox handles auth and GitHub handles
enforcement, the agent's job is almost entirely "translate between how a
human describes a problem and which two or three tool calls solve it."


## 3 / How to get started

The full sample ? code, README, and setup steps ? is
[`18-skill-review-teams-bot`](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/hosted-agents/agent-framework/responses/18-skill-review-teams-bot)
in the Foundry samples repo. Here's the short version:

1. **Create the GitHub project connection and toolbox once**, per project
   (see the sample README for the exact PUT/POST bodies):
   - An `OAuth2` / `catalog_MCP` project connection targeting
     `https://api.githubcopilot.com/mcp`.
   - A toolbox that attaches that connection, promoted to `default`.
2. **Write the agent.** Construct a `FoundryToolbox(credential)`, pass it
   as the agent's `tools`, and write instructions in terms of what your
   users are trying to do, not which git operations exist. Keep a
   glossary of one-line, plain-language explanations for
   branch/commit/PR/diff/merge/fork ? you'll want it the first time
   someone asks.
3. **Deploy with `azd deploy`.** Standard Agent Framework hosted-agent
   flow ? no extra steps for the toolbox, since it's a project-level
   resource the agent just references by name.
4. **Publish to Teams** from the Foundry portal (**Publish -> Publish to
   Teams and Microsoft 365**). Business users sign in once for the bot
   itself, and once for GitHub the first time they trigger a GitHub
   action ? after that, it's just chat.
5. **Set up branch protection** on your skills repo: require an approving
   review before merge. This is what actually enforces "two people must
   agree," not anything the agent says.

If you're building anything similar ? any Teams-facing agent that needs
to act on a developer-facing system on behalf of a business user ? the
pattern generalizes past GitHub: put a per-user OAuth2/catalog-MCP
connection in front of it, let the platform resolve identity per caller,
and keep your agent's instructions focused on translating intent rather
than asking users to speak the underlying system's language.


> **Merging a PR is not the same as publishing.** Approving and merging a
> skill's pull request only updates the file in GitHub -- it does not, by
> itself, change what any running Foundry agent does. This sample's agent
> exposes a second, explicit capability -- `publish_skill_to_foundry` -- that
> a user can ask for once a change is merged (e.g. "make the refund-policy
> skill live"), which re-reads the merged file and calls Foundry's
> `beta.skills` API to create/update the live skill. It runs under the
> hosted agent's own managed identity (not the requesting user's GitHub
> identity), since it writes to the Foundry project, not to GitHub.


## 4 / Try it yourself -- the full, runnable sample

Everything below materializes the actual sample on disk (via `%%writefile`)
so you can run it end to end: create the toolbox, deploy the agent, and get
a Teams app you can sideload immediately. This is the same code referenced
above, kept here so the recipe is self-contained and downloadable straight
from this notebook -- no separate repo checkout required.

Run the cells in order. They assume:

- The [Azure Developer CLI](https://aka.ms/azd) with the Foundry agent
  extension: `azd extension install azure.ai.agents`
- An existing Microsoft Foundry project with at least one model deployment
- `azd auth login` already completed


In [ ]:
import os

sample_dir = "skill-review-teams-bot"
os.makedirs(sample_dir, exist_ok=True)
os.chdir(sample_dir)
print("Working directory:", os.getcwd())


### `main.py` -- the agent

In [ ]:
%%writefile main.py
# Copyright (c) Microsoft. All rights reserved.

"""Skill-review Teams bot.

Publishes an Agent Framework agent to Teams (via the Foundry portal's
"Publish to Teams and Microsoft 365" flow) so business users can, from a
normal Teams chat:

  * List open Foundry Skill pull requests in a GitHub repo.
  * Read a PR's description and file diff before deciding.
  * Comment on, approve, or request changes on a PR.
  * Merge a PR once it has the required approval.
  * Propose their own skill change by creating a branch, committing an
    edited/new ``SKILL.md`` (or other skill file), and opening a PR for
    review by other business users.
  * Publish an already-merged, approved ``SKILL.md`` as a live Foundry
    Skill, once someone explicitly asks to make it "live"/"active".

Merging a skill PR only updates the file in the GitHub repo -- it does not
by itself change anything in Foundry. Publishing is a second, explicit step
(``publish_skill_to_foundry``) that re-reads the merged file and uploads it
via the project's ``beta.skills`` API using the *agent's own* managed
identity (not the requesting user's GitHub identity), since it writes to
the Foundry project rather than to GitHub.

GitHub access goes through a Foundry **toolbox** wired to GitHub's catalog
MCP server (``https://api.githubcopilot.com/mcp``) via an ``OAuth2`` /
``catalog_MCP`` project connection, instead of a single shared
``GITHUB_TOKEN``. Each Teams user consents to GitHub once (the toolbox
returns a per-user OAuth link the first time they trigger a GitHub tool
call), and every subsequent PR comment/review/merge/branch/commit made in
this chat is executed -- and attributed on GitHub -- as *that* user, not a
shared service identity. This matters because GitHub branch-protection
"required reviewers" checks, and the audit trail on every skill PR, only
count actions taken by the authenticated identity making the API call.

Because GitHub access is now a generic MCP toolbox rather than custom
Python wrappers, the previous code-enforced "no merge without an approving
review" gate can no longer intercept the call before it reaches GitHub.
Two layers replace it:

  1. The agent's instructions require it to read the PR's reviews (via the
     toolbox's own ``pull_request_read`` tool) before ever calling the
     merge tool, and to refuse if there is no approval or an outstanding
     "request changes" review.
  2. For a hard, unbypassable guarantee, configure GitHub branch
     protection on the target repo ("Require a pull request before
     merging" + "Require approvals") -- GitHub itself then rejects the
     merge call regardless of what the model decides to do.

See README.md for how the toolbox and its GitHub OAuth2/catalog_MCP
connection are created (project connection + toolbox attach + consent).
"""

import asyncio
import json
import logging
import os
import re
import sys

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient, FoundryToolbox, ResponsesHostServer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

logger = logging.getLogger(__name__)

# ``azure-ai-projects`` only exposes ``beta.skills`` (the Foundry Skills API)
# starting at 2.7.0, but ``agent-framework-foundry`` pins
# ``azure-ai-projects<2.7.0`` -- the two cannot be installed in the same
# environment. ``publish_skill_worker.py`` runs in a second, isolated venv
# (built in the Dockerfile) that has a modern azure-ai-projects and nothing
# else; we shell out to it instead of importing azure-ai-projects here.
_SKILLS_VENV_PYTHON = os.environ.get("SKILLS_VENV_PYTHON", "/opt/skills-venv/bin/python")
_WORKER_SCRIPT = os.path.join(os.path.dirname(__file__), "publish_skill_worker.py")


async def publish_skill_to_foundry(skill_name: str, skill_markdown: str) -> str:
    """Create or update a live Foundry Skill from a SKILL.md file's full text.

    Call this only after the user has explicitly asked to make an already
    reviewed-and-merged skill change take effect in Foundry (e.g. "make this
    live", "publish this skill", "activate the refund-policy skill") -- not
    merely to merge its GitHub PR, which only updates the file in git.
    ``skill_name`` must match the ``name:`` value in the file's own YAML
    front matter. ``skill_markdown`` must be the exact, current text of the
    merged SKILL.md (read it from GitHub first; never invent or paraphrase
    it). This runs as the agent's own Foundry identity, not the calling
    user's GitHub identity, so it needs no GitHub OAuth consent.
    """
    description = _extract_frontmatter_field(skill_markdown, "description") or skill_name
    instructions = _strip_frontmatter(skill_markdown)

    python = _SKILLS_VENV_PYTHON if os.path.exists(_SKILLS_VENV_PYTHON) else sys.executable
    process = await asyncio.create_subprocess_exec(
        python,
        _WORKER_SCRIPT,
        stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
        env=os.environ,
    )
    request = json.dumps(
        {"skill_name": skill_name, "description": description, "instructions": instructions}
    ) + "\n"
    stdout, stderr = await process.communicate(request.encode("utf-8"))

    try:
        result = json.loads(stdout.decode("utf-8").strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        raise RuntimeError(
            f"publish_skill_worker.py produced no parseable output "
            f"(exit={process.returncode}): {stderr.decode('utf-8', errors='replace')[:500]}"
        )

    if not result.get("ok"):
        raise RuntimeError(f"Failed to publish skill '{skill_name}': {result.get('error')}")
    return f"Published skill '{skill_name}' to Foundry (version={result['version']})."


def _extract_frontmatter_field(markdown: str, field: str) -> str | None:
    """Pull a single ``field: value`` line out of a SKILL.md YAML front matter block."""
    match = re.search(r"^---\s*\n(.*?)\n---\s*\n", markdown, re.DOTALL)
    if not match:
        return None
    frontmatter = match.group(1)
    field_match = re.search(rf"^{field}:\s*(.+)$", frontmatter, re.MULTILINE)
    return field_match.group(1).strip().strip('"').strip("'") if field_match else None


def _strip_frontmatter(markdown: str) -> str:
    """Return the markdown body of a SKILL.md file with its YAML front matter removed."""
    match = re.match(r"^---\s*\n.*?\n---\s*\n", markdown, re.DOTALL)
    return markdown[match.end():].lstrip("\n") if match else markdown


def _repo() -> str:
    repo = os.environ.get("GITHUB_REPO", "").strip()
    if not repo or "/" not in repo:
        raise RuntimeError("GITHUB_REPO must be set to 'owner/repo'.")
    return repo


def _skill_label() -> str:
    return os.environ.get("SKILL_PR_LABEL", "skill").strip() or "skill"


def _skills_base_path() -> str:
    return os.environ.get("SKILLS_BASE_PATH", "skills/").strip().strip("/")


def main() -> None:
    credential = DefaultAzureCredential()

    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=credential,
    )

    # The toolbox endpoint/name is resolved from FOUNDRY_PROJECT_ENDPOINT +
    # TOOLBOX_NAME (defaulting to "github-review-tb"; see README). The hosting
    # server connects this toolbox on first use and forwards each request's
    # platform call-id, so the Foundry MCP proxy resolves GitHub OAuth state
    # per calling Teams user automatically -- no per-user wiring needed here.
    os.environ.setdefault("TOOLBOX_NAME", "github-review-tb")
    toolbox = FoundryToolbox(credential)

    owner_repo = _repo()
    owner, repo = owner_repo.split("/", 1)
    label = _skill_label()
    base_path = _skills_base_path()

    agent = Agent(
        client=client,
        instructions=(
            "You help business users review and propose changes to Foundry Skills "
            f"stored as pull requests in the GitHub repository '{owner_repo}' "
            f"(owner='{owner}', repo='{repo}'). Skill-change PRs are labeled "
            f"'{label}'; new or edited skill files must live under '{base_path}/'. "
            "GitHub tools are provided by a connected toolbox (names prefixed "
            "'gh___'), each call acting as the individual Teams user's own GitHub "
            "identity -- if a tool call returns a CONSENT_REQUIRED error with a URL, "
            "tell the user to open that URL to sign in to GitHub once, then retry "
            "their request.\n\n"
            "IMPORTANT -- your users are business users, not developers. They do "
            "not know git/GitHub terminology (branch, commit, pull request, merge, "
            "fork, diff, review) and you must never ask them to choose one of these "
            "operations by name or make them pick which git primitive to use. "
            "Instead: listen to what they are trying to accomplish in plain "
            "language (e.g. 'I want to fix the refund policy skill', 'has anyone "
            "looked at my change yet', 'make this change live', 'undo what I "
            "proposed'), silently map that intent onto the right sequence of "
            "gh___ tool calls yourself, and describe what you did and what happens "
            "next in plain language, not git jargon. If a user's intent is genuinely "
            "ambiguous (e.g. it's unclear which skill or which PR they mean), ask a "
            "clarifying question about *their goal*, not about which git operation "
            "to run.\n\n"
            "If a user asks what a git/GitHub term means, or seems unsure what a "
            "step you just took actually did, explain it in plain, non-technical "
            "language, for example:\n"
            "  - branch: 'a private draft copy of the skills where you can make "
            "changes without affecting anyone else until you're ready to share "
            "them.'\n"
            "  - commit: 'saving a snapshot of your edit with a short note about "
            "what changed.'\n"
            "  - pull request (PR): 'a request to have your proposed change "
            "reviewed and, once approved, included in the shared, live version.'\n"
            "  - diff: 'a side-by-side view of exactly what text changed.'\n"
            "  - review / approve / request changes: 'another person reading your "
            "proposed change and either signing off on it or asking for edits "
            "first.'\n"
            "  - merge: 'combining an approved change into the shared, "
            "reviewed version everyone else edits from -- it does not by "
            "itself change how any running assistant behaves until someone "
            "also asks to publish/activate it.'\n"
            "  - fork: 'your own separate copy of the whole project, used instead "
            "of a branch when you don't have direct write access to the original.'\n"
            "Keep these explanations short and only give them when they are asked "
            "for or clearly needed to understand what just happened -- do not "
            "lecture on git mechanics unprompted.\n\n"
            "When asked to review a PR, always show its description and diff "
            "(gh___pull_request_read) before recommending approve/request-changes, "
            "described in plain terms (what changed and why it matters), not raw "
            "diff syntax. Never call gh___pull_request_review_write or "
            "gh___merge_pull_request without the user's explicit decision stated "
            "in the conversation -- but phrase your check as 'Should I approve "
            "this?' / 'Ready for me to make this change live?', not 'Should I call "
            "merge_pull_request?'. Before calling gh___merge_pull_request, first "
            "call gh___pull_request_read to check the PR's reviews yourself: "
            "refuse to merge (and tell the user why, in plain language) if there "
            "is no approving review, or if there is an outstanding 'request "
            "changes' review that hasn't been superseded by a later approval. "
            "GitHub will also reject a merge or a self-review outright (e.g. you "
            "cannot approve your own PR) -- explain that plainly too (e.g. 'GitHub "
            "requires someone else to approve this before it can go live') rather "
            "than retrying.\n\n"
            "Merging is not the same as publishing. Merging only updates the "
            "file everyone edits from next; it does not change what any "
            "running assistant actually does. If a user asks to make an "
            "already-merged, approved skill change 'live', 'active', or "
            "'published' *in Foundry* (as opposed to just 'merged'/'approved' "
            "in GitHub -- ask which they mean if it's unclear), first confirm "
            "the PR is merged (merging it yourself first if needed and "
            "allowed, per the rules above), then call gh___get_file_contents "
            f"to read the exact, current text of that skill's SKILL.md under "
            f"'{base_path}/' on the default branch, then call "
            "publish_skill_to_foundry with the skill's name (from its own "
            "YAML front matter) and that exact text -- never invent, "
            "summarize, or paraphrase the skill text yourself. Tell the user "
            "in plain language once it's live for Foundry agents to use.\n\n"
            "When a user wants to propose a change, confirm with them (in plain "
            "language) only the two things that actually require their input: "
            "which skill they mean, and what the change should say. Never ask "
            "them for a branch name, commit message, or PR title/description -- "
            "always generate all of those yourself from the skill name and a "
            "short summary of the change (e.g. branch "
            "'skill/<skill-name>-<short-slug>', a commit message and PR title "
            "that plainly describe what changed, and a PR description that "
            "restates the user's requested change in a sentence or two). Then use "
            "gh___create_branch, gh___create_or_update_file (or "
            f"gh___push_files), and gh___create_pull_request to open a PR against "
            "main from that branch (gh___create_pull_request has no labels "
            "parameter, only 'reviewers' -- do not pass a labels argument to it "
            "or the call will fail). Immediately after, call gh___issue_write with "
            "method='update', the new PR's number as issue_number, and "
            f"labels=['{label}'] to apply the required '{label}' label -- this is "
            "a separate, required step, not part of create_pull_request. Do not "
            "attempt to assign a specific reviewer on the PR (there is no "
            "reliable way to know who should review it, and requests to assign "
            "one often fail); if the user asks who should review it, tell them "
            "any of their teammates with repo access can look at it. Tell "
            "the user what you did and who needs to review it next, without "
            "narrating each git call or the "
            "names you chose unless they ask. Keep answers concise and "
            "Teams-chat friendly."
        ),
        tools=[toolbox, publish_skill_to_foundry],
        default_options={"store": True},
    )

    server = ResponsesHostServer(agent)
    server.run()


if __name__ == "__main__":
    main()


### `publish_skill_worker.py` -- isolated-venv helper for the Foundry Skills API

The Foundry Skills API (`beta.skills.*`) needs `azure-ai-projects>=2.7.0`, but `agent-framework-foundry` pins `azure-ai-projects<2.7.0` -- the two can't share one Python environment. `main.py` shells out to this small standalone script, which runs in a second, isolated venv (built in the `Dockerfile` below) that has only the modern `azure-ai-projects` installed.


In [ ]:
%%writefile publish_skill_worker.py
# Copyright (c) Microsoft. All rights reserved.

"""Isolated-venv worker that calls the Foundry Skills API.

``azure-ai-projects`` only exposes ``beta.skills`` (the Foundry Skills API)
starting at 2.7.0, but ``agent-framework-foundry`` pins
``azure-ai-projects<2.7.0``. Both packages cannot be installed together in
the same environment. This script is installed into a *second*, isolated
virtual environment (see ``Dockerfile``) that has a modern
``azure-ai-projects`` and nothing else, and is invoked as a subprocess by
``main.py`` -- so the main app's dependencies never see the conflict.

Reads a single-line JSON object from stdin: ``{"skill_name": str,
"description": str, "instructions": str}``. Prints a single-line JSON
result to stdout: ``{"ok": true, "version": <int>}`` or
``{"ok": false, "error": str}``, and exits 0/1 accordingly.
"""

import asyncio
import json
import os
import sys


async def _publish(skill_name: str, description: str, instructions: str) -> int:
    from azure.ai.projects.aio import AIProjectClient
    from azure.ai.projects.models import SkillInlineContent
    from azure.identity.aio import DefaultAzureCredential

    endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
    async with (
        DefaultAzureCredential() as credential,
        AIProjectClient(endpoint=endpoint, credential=credential, allow_preview=True) as project,
    ):
        version = await project.beta.skills.create(
            skill_name,
            inline_content=SkillInlineContent(description=description, instructions=instructions),
            default=True,
        )
        print(json.dumps({"ok": True, "version": version.version}))
        return 0


def main() -> int:
    try:
        request = json.loads(sys.stdin.readline())
        return asyncio.run(
            _publish(request["skill_name"], request["description"], request["instructions"])
        )
    except Exception as exc:  # noqa: BLE001 - surface any failure to the parent process as JSON
        print(json.dumps({"ok": False, "error": f"{type(exc).__name__}: {exc}"}))
        return 1


if __name__ == "__main__":
    sys.exit(main())



### `toolbox.yaml` -- the GitHub connection this agent consumes

In [ ]:
%%writefile toolbox.yaml
# toolbox.yaml
# Defines the toolbox this agent consumes. Create it once the connection
# below exists (setup.ps1 / setup.sh do both steps for you):
#   azd ai connection create github-oauth --kind remote-tool \
#     --target https://api.githubcopilot.com/mcp --auth-type oauth2 \
#     --connector-name foundrygithubmcp
#   azd ai toolbox create github-review-tb --from-file ./toolbox.yaml
# The toolbox's MCP endpoint is written to the azd environment as
# TOOLBOX_GITHUB_REVIEW_TB_MCP_ENDPOINT; TOOLBOX_NAME=github-review-tb is
# all the agent needs to resolve it at runtime.
description: Per-user GitHub access for reviewing and proposing Foundry Skill PRs
connections:
  - name: github-oauth


### `agent.yaml` -- hosted agent manifest

In [ ]:
%%writefile agent.yaml
# yaml-language-server: $schema=https://raw.githubusercontent.com/microsoft/AgentSchema/refs/heads/main/schemas/v1.0/ContainerAgent.yaml
kind: hosted
name: agent-framework-agent-skill-review-teams-bot
protocols:
  - protocol: responses
    version: 1.0.0
resources:
  cpu: '0.25'
  memory: '0.5Gi'
environment_variables:
  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
    value: ${AZURE_AI_MODEL_DEPLOYMENT_NAME}
  - name: GITHUB_REPO
    value: ${GITHUB_REPO}
  - name: SKILL_PR_LABEL
    value: ${SKILL_PR_LABEL}
  - name: SKILLS_BASE_PATH
    value: ${SKILLS_BASE_PATH}
  - name: TOOLBOX_NAME
    value: ${TOOLBOX_NAME}


### `requirements.txt`

In [ ]:
%%writefile requirements.txt
# Use the narrow Foundry subpackages to keep dependencies light.
agent-framework-foundry
agent-framework-foundry-hosting

# NOTE: azure-ai-projects is intentionally NOT listed here. The Foundry
# Skills API used by publish_skill_to_foundry needs azure-ai-projects>=2.7.0,
# which conflicts with agent-framework-foundry's own azure-ai-projects<2.7.0
# pin. That dependency is installed into a separate, isolated venv
# (/opt/skills-venv, see Dockerfile) and called via subprocess from
# publish_skill_worker.py instead of being imported directly in this app.

# debugpy enables local debugging of this agent with the Foundry Toolkit VS Code extension.
debugpy



### `Dockerfile` -- builds the isolated venv for the Skills API


In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then \
        pip install -r requirements.txt; \
    else \
        echo "No requirements.txt found"; \
    fi

# Isolated venv for publish_skill_worker.py: azure-ai-projects>=2.7.0 (needed
# for the Foundry Skills API) conflicts with agent-framework-foundry's own
# azure-ai-projects<2.7.0 pin, so it cannot share the main app's environment.
RUN python -m venv /opt/skills-venv \
    && /opt/skills-venv/bin/pip install --no-cache-dir "azure-ai-projects>=2.7.0" azure-identity
ENV SKILLS_VENV_PYTHON=/opt/skills-venv/bin/python

EXPOSE 8088

CMD ["python", "main.py"]


### `.env.example`

In [ ]:
%%writefile .env.example
FOUNDRY_PROJECT_ENDPOINT="..."
AZURE_AI_MODEL_DEPLOYMENT_NAME="..."
GITHUB_REPO="contoso/foundry-skills"
SKILL_PR_LABEL="skill"
SKILLS_BASE_PATH="skills/"

# Name of the Foundry toolbox (see README) wired to a GitHub OAuth2/catalog_MCP
# project connection. GitHub access is per-Teams-user OAuth via this toolbox,
# not a shared GITHUB_TOKEN.
TOOLBOX_NAME="github-review-tb"


### `setup.ps1` / `setup.sh` -- one command from clone to sideloadable Teams app

These wrap four `azd ai` calls: create the GitHub OAuth2 connection, create
the toolbox from `toolbox.yaml`, deploy the agent, then build a Teams app
package (`azd ai agent pack`) and a no-admin-approval shareable install link
(`azd ai agent publish --scope shared`).

In [ ]:
%%writefile setup.ps1
#!/usr/bin/env pwsh
<#
.SYNOPSIS
  One-command setup: creates the GitHub OAuth2 connection + toolbox,
  deploys this agent, and builds a Teams app package you can sideload
  immediately.

.DESCRIPTION
  Run this from an azd environment that already has FOUNDRY_PROJECT_ENDPOINT
  set (azd env set FOUNDRY_PROJECT_ENDPOINT "https://<account>.services.ai
  .azure.com/api/projects/<project>"). It does not provision Azure resources
  for you -- point it at an existing Foundry project.

.PARAMETER GithubRepo
  owner/repo containing the Foundry Skill files this bot will review, e.g.
  contoso/foundry-skills.

.EXAMPLE
  ./setup.ps1 -GithubRepo contoso/foundry-skills
#>
param(
  [Parameter(Mandatory = $true)]
  [string]$GithubRepo,

  [string]$ConnectionName = "github-oauth",
  [string]$ToolboxName = "github-review-tb",
  [switch]$SkipPublish
)

$ErrorActionPreference = "Stop"

function Step($msg) { Write-Host "`n==> $msg" -ForegroundColor Cyan }

Step "Creating GitHub OAuth2 project connection '$ConnectionName' (per-user consent, no shared token)"
azd ai connection create $ConnectionName `
  --kind remote-tool `
  --target https://api.githubcopilot.com/mcp `
  --auth-type oauth2 `
  --connector-name foundrygithubmcp `
  --force
if ($LASTEXITCODE -ne 0) { throw "connection create failed" }

Step "Creating toolbox '$ToolboxName' from toolbox.yaml"
azd ai toolbox create $ToolboxName --from-file "$PSScriptRoot/toolbox.yaml" --force
if ($LASTEXITCODE -ne 0) { throw "toolbox create failed" }

Step "Setting agent environment variables (GITHUB_REPO, TOOLBOX_NAME)"
azd env set GITHUB_REPO $GithubRepo
azd env set TOOLBOX_NAME $ToolboxName

Step "Deploying the agent"
azd deploy
if ($LASTEXITCODE -ne 0) { throw "azd deploy failed" }

Step "Building a Teams app package you can sideload right now"
azd ai agent pack --scope personal
if ($LASTEXITCODE -ne 0) { throw "azd ai agent pack failed" }

if (-not $SkipPublish) {
  Step "Publishing a shareable Teams app link (no tenant-admin approval needed)"
  azd ai agent publish --scope shared
}

Write-Host "`nDone. Sideload the appPackage.zip written above in Teams (Apps -> Manage your apps -> Upload an app -> Upload a custom app), or share the published link from the previous step." -ForegroundColor Green
Write-Host "The first GitHub action any user takes will prompt them to sign in with their own GitHub account -- that's expected." -ForegroundColor Green


In [ ]:
%%writefile setup.sh
#!/usr/bin/env bash
# One-command setup: creates the GitHub OAuth2 connection + toolbox, deploys
# this agent, and builds a Teams app package you can sideload immediately.
#
# Run this from an azd environment that already has FOUNDRY_PROJECT_ENDPOINT
# set (azd env set FOUNDRY_PROJECT_ENDPOINT "https://<account>.services.ai
# .azure.com/api/projects/<project>"). It does not provision Azure resources
# for you -- point it at an existing Foundry project.
#
# Usage: ./setup.sh <owner/repo> [--skip-publish]

set -euo pipefail

GITHUB_REPO="${1:?Usage: ./setup.sh <owner/repo> [--skip-publish]}"
SKIP_PUBLISH="${2:-}"
CONNECTION_NAME="${CONNECTION_NAME:-github-oauth}"
TOOLBOX_NAME="${TOOLBOX_NAME:-github-review-tb}"
SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"

step() { printf '\n==> %s\n' "$1"; }

step "Creating GitHub OAuth2 project connection '$CONNECTION_NAME' (per-user consent, no shared token)"
azd ai connection create "$CONNECTION_NAME" \
  --kind remote-tool \
  --target https://api.githubcopilot.com/mcp \
  --auth-type oauth2 \
  --connector-name foundrygithubmcp \
  --force

step "Creating toolbox '$TOOLBOX_NAME' from toolbox.yaml"
azd ai toolbox create "$TOOLBOX_NAME" --from-file "$SCRIPT_DIR/toolbox.yaml" --force

step "Setting agent environment variables (GITHUB_REPO, TOOLBOX_NAME)"
azd env set GITHUB_REPO "$GITHUB_REPO"
azd env set TOOLBOX_NAME "$TOOLBOX_NAME"

step "Deploying the agent"
azd deploy

step "Building a Teams app package you can sideload right now"
azd ai agent pack --scope personal

if [ "$SKIP_PUBLISH" != "--skip-publish" ]; then
  step "Publishing a shareable Teams app link (no tenant-admin approval needed)"
  azd ai agent publish --scope shared
fi

printf '\nDone. Sideload the appPackage.zip written above in Teams (Apps -> Manage your apps -> Upload an app -> Upload a custom app), or share the published link from the previous step.\n'
printf 'The first GitHub action any user takes will prompt them to sign in with their own GitHub account -- that is expected.\n'


### Run it

```bash
azd env new skill-review-bot
azd env set FOUNDRY_PROJECT_ENDPOINT "https://<account>.services.ai.azure.com/api/projects/<project>"

# Windows/PowerShell
./setup.ps1 -GithubRepo contoso/foundry-skills

# macOS/Linux
chmod +x setup.sh && ./setup.sh contoso/foundry-skills
```

That prints an `appPackage.zip` you can sideload in Teams (**Apps -> Manage
your apps -> Upload an app -> Upload a custom app**), plus a shareable
install link from `azd ai agent publish`. The first GitHub-touching command
any Teams user runs will prompt them to sign in with their own GitHub
account once -- that's the per-user OAuth consent working as intended, not
an error.

Before pointing real users at it, turn on branch protection on your skills
repo (require an approving review before merge) -- that's the layer that
actually enforces "two people must agree," not anything the agent's
instructions say.
